In [ ]:
# ============================================================
# VALIDATION A — BRANCH A
# D10 — IMPI — Inquérito Mensal à Produção Industrial
# ============================================================
#
# Methodology stage covered:
# Stage 4 — Post-processing and Validation
#
# Compares the preserved Branch A extraction against the
# fixed Stage 1 document-grounded reference dataset.
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import hashlib
import json
import re
import unicodedata

import pandas as pd


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D10"
DOCUMENT_NAME = "IMPI — Inquérito Mensal à Produção Industrial"

BRANCH = "A"
BRANCH_NAME = "Direct Ingestion"
INPUT_REPRESENTATION = "Original PDF questionnaire"

EXPECTED_SOURCE_SHA256 = (
    "fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5"
)

EXPECTED_RECORD_COUNT = 69

EXPECTED_CATEGORY_COUNTS = {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11,
}

FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Code",
    "Expected Value Type",
    "Source Location",
]

MANDATORY_STRING_FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Expected Value Type",
    "Source Location",
]

NULLABLE_STRING_FIELDS = ["Code"]

BASE_IDENTITY_FIELDS = ["Category", "Field or Concept"]

DUPLICATE_DISAMBIGUATION_FIELD = "Section"

ALIGNMENT_IDENTITY_FIELDS = [
    "Category",
    "Field or Concept",
    "Section",
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Code",
    "Expected Value Type",
    "Source Location",
]

DIAGNOSTIC_FIELDS = ["Section", "Description"]

OUTPUT_DIR = Path("outputs_D10_validation_A_revised")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)
print("Output directory:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 2. Upload canonical validation inputs
# ============================================================

print(
    "Upload exactly four files:\n"
    "1. D10_reference_values.csv\n"
    "2. D10_branch_A_parsed_extraction.json\n"
    "3. D10_branch_A_technical_diagnostics.json\n"
    "4. D10_branch_A_experiment_metadata.json"
)

uploaded = files.upload()

required_names = {
    "D10_reference_values.csv",
    "D10_branch_A_parsed_extraction.json",
    "D10_branch_A_technical_diagnostics.json",
    "D10_branch_A_experiment_metadata.json",
}

observed_names = set(uploaded.keys())

if observed_names != required_names:
    raise ValueError(
        "Upload exactly the four canonical files listed above. "
        f"Observed: {sorted(observed_names)}"
    )

REFERENCE_PATH = Path("D10_reference_values.csv")
EXTRACTION_PATH = Path("D10_branch_A_parsed_extraction.json")
TECHNICAL_DIAGNOSTICS_PATH = Path("D10_branch_A_technical_diagnostics.json")
METADATA_PATH = Path("D10_branch_A_experiment_metadata.json")


In [ ]:
# ============================================================
# 3. Load inputs and verify provenance
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
TECHNICAL_DIAGNOSTICS_SHA256 = sha256_file(TECHNICAL_DIAGNOSTICS_PATH)
METADATA_SHA256 = sha256_file(METADATA_PATH)


reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=True,
)

reference_df = reference_df.where(
    pd.notna(reference_df),
    None,
)

parsed_extraction = json.loads(
    EXTRACTION_PATH.read_text(encoding="utf-8")
)

if not isinstance(parsed_extraction, dict):
    raise ValueError(
        "Canonical parsed extraction must be a top-level JSON object."
    )

document_id_correct = (
    parsed_extraction.get("document_id") == DOCUMENT_ID
)

branch_correct = (
    parsed_extraction.get("branch") == BRANCH
)

extracted_records = parsed_extraction.get("records")

records_is_list = isinstance(
    extracted_records,
    list
)

if not records_is_list:
    raise ValueError(
        "Parsed extraction must contain a records list."
    )

extracted_df = pd.DataFrame(
    extracted_records
)


technical_diagnostics = json.loads(
    TECHNICAL_DIAGNOSTICS_PATH.read_text(encoding="utf-8")
)

experiment_metadata = json.loads(
    METADATA_PATH.read_text(encoding="utf-8")
)


branch_a_structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

parsed_extraction_hash_matches_metadata = (
    experiment_metadata.get(
        "parsed_extraction_sha256"
    )
    == EXTRACTION_SHA256
)

source_hash_matches_stage_1 = (
    experiment_metadata.get(
        "source_sha256"
    )
    == EXPECTED_SOURCE_SHA256
)


print("Reference SHA-256:", REFERENCE_SHA256)
print("Parsed extraction SHA-256:", EXTRACTION_SHA256)
print(
    "Branch A structurally evaluable:",
    branch_a_structurally_evaluable,
)

print(
    "Parsed extraction hash matches metadata:",
    parsed_extraction_hash_matches_metadata,
)
print(
    "Source hash matches Stage 1:",
    source_hash_matches_stage_1,
)


In [ ]:
# ============================================================
# 4. Schema, type and content diagnostics
# ============================================================

reference_schema_exact = (
    reference_df.columns.tolist() == FIELDS
)

extraction_schema_exact = (
    extracted_df.columns.tolist() == FIELDS
)


def validate_record_types(records, dataset_name):
    issues = []

    for index, record in enumerate(records):

        if not isinstance(record, dict):
            issues.append({
                "dataset": dataset_name,
                "record_index": index,
                "field": None,
                "issue": "Record is not an object",
            })
            continue

        observed_fields = list(record.keys())

        # Field order is a diagnostic only.
        if set(observed_fields) != set(FIELDS):
            issues.append({
                "dataset": dataset_name,
                "record_index": index,
                "field": None,
                "issue": "Field set differs from schema",
                "observed_fields": observed_fields,
            })

        for field in MANDATORY_STRING_FIELDS:
            value = record.get(field)

            if value is None or (
                isinstance(value, str)
                and not value.strip()
            ):
                issues.append({
                    "dataset": dataset_name,
                    "record_index": index,
                    "field": field,
                    "issue": "Missing mandatory value",
                })

            elif not isinstance(value, str):
                issues.append({
                    "dataset": dataset_name,
                    "record_index": index,
                    "field": field,
                    "issue": "Expected string",
                    "observed_type": type(value).__name__,
                })

        code_value = record.get("Code")

        if (
            code_value is not None
            and not isinstance(code_value, str)
        ):
            issues.append({
                "dataset": dataset_name,
                "record_index": index,
                "field": "Code",
                "issue": "Expected string or null",
                "observed_type": type(code_value).__name__,
            })

    return issues


reference_type_issues = validate_record_types(
    reference_df.to_dict("records"),
    "Reference",
)

extraction_type_issues = validate_record_types(
    extracted_records,
    "Extraction",
)

reference_types_valid = (
    len(reference_type_issues) == 0
)

extraction_types_valid = (
    len(extraction_type_issues) == 0
)


reference_record_count_valid = (
    len(reference_df) == EXPECTED_RECORD_COUNT
)

extraction_record_count_valid = (
    len(extracted_df) == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

extraction_category_counts = (
    extracted_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

extraction_category_counts_valid = (
    extraction_category_counts
    == EXPECTED_CATEGORY_COUNTS
)


schema_validity = bool(
    branch_a_structurally_evaluable
)

local_schema_diagnostics = {
    "document_id_correct":
        bool(document_id_correct),
    "branch_correct":
        bool(branch_correct),
    "records_is_list":
        bool(records_is_list),
    "extraction_schema_exact":
        bool(extraction_schema_exact),
    "extraction_types_valid":
        bool(extraction_types_valid),
}

print("Reference schema exact:", reference_schema_exact)
print("Extraction schema exact:", extraction_schema_exact)
print("Reference types valid:", reference_types_valid)
print("Extraction types valid:", extraction_types_valid)
print("Schema validity:", schema_validity)
print("Reference record count:", len(reference_df))
print("Extraction record count:", len(extracted_df))
print("Reference category counts:", reference_category_counts)
print("Extraction category counts:", extraction_category_counts)


In [ ]:
# ============================================================
# 5. Confirm current Stage 1 D10 reference semantics
# ============================================================

EXPECTED_UAE_TEMPLATE_LABELS = {
    "Código da UAE",
    "Designação da UAE",
    "Situação da UAE perante a atividade",
    "Observações da UAE",
    "Confirmar",
    "Produtos",
}

EXPECTED_PRODUCT_TABLE_LABELS = {
    "NIF",
    "UAE",
    "Período de Referência",
    "Nº",
    "Produto",
    "Unid.",
    "Código",
    "Quantidades produzidas",
    "Quantidades vendidas",
    "Valor das vendas / prestação de serviços",
    "Observações empresa",
    "Observações INE",
}


observed_uae_template_labels = set(
    reference_df.loc[
        reference_df["Category"]
        == "UAE template element",
        "Field or Concept",
    ]
)

observed_product_table_labels = set(
    reference_df.loc[
        reference_df["Category"]
        == "Product table field",
        "Field or Concept",
    ]
)


reference_period_field_valid = (
    (
        reference_df["Field or Concept"]
        == "Referência dos dados"
    ).sum()
    == 1
)

source_location_pattern_valid = bool(
    reference_df["Source Location"]
    .fillna("")
    .str.match(r"^PDF page [1-4] — .+$")
    .all()
)


reference_semantic_checks = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "reference_category_counts_valid":
        bool(reference_category_counts_valid),

    "reference_schema_exact":
        bool(reference_schema_exact),

    "reference_types_valid":
        bool(reference_types_valid),

    "uae_template_valid":
        observed_uae_template_labels
        == EXPECTED_UAE_TEMPLATE_LABELS,

    "reference_period_field_valid":
        bool(reference_period_field_valid),

    "product_table_structurally_evaluable":
        observed_product_table_labels
        == EXPECTED_PRODUCT_TABLE_LABELS,

    "source_location_pattern_valid":
        bool(source_location_pattern_valid),
}


reference_semantics_valid = all(
    reference_semantic_checks.values()
)

print(
    json.dumps(
        reference_semantic_checks,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "Corrected/current D10 reference semantics valid:",
    reference_semantics_valid,
)

if not reference_semantics_valid:
    raise AssertionError(
        "The supplied D10 reference does not match the current "
        "frozen Stage 1 D10 reference semantics."
    )


In [ ]:
# ============================================================
# 6. Comparison-only normalisation
# ============================================================

def normalise_text(value):
    if value is None:
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = (
        text
        .replace("—", "-")
        .replace("–", "-")
        .replace("‑", "-")
        .replace("“", '"')
        .replace("”", '"')
        .replace("’", "'")
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text.casefold()


def normalise_identity_text(value):
    text = normalise_text(value)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


SECTION_EQUIVALENCE_GROUPS = [
    {
        "header",
        "questionnaire header",
        "legal notice",
    },
    {
        "response contacts",
        "contact and response information",
        "response information",
    },
    {
        "reference data",
        "reference-data area",
    },
    {
        "identification of statistical unit",
        "section i",
        "i - identificação da unidade estatística",
        "identificação da unidade estatística",
    },
    {
        "activity status",
        "section ii",
        "ii - situação da unidade estatística no período de referência dos dados",
        "situação da unidade estatística no período de referência dos dados",
    },
    {
        "observations",
        "section iii",
        "iii - observações",
        "observações",
    },
    {
        "responsible person",
        "section iv",
        "iv - responsável pelo preenchimento",
        "responsável pelo preenchimento",
    },
    {
        "uae information",
        "uae template",
    },
    {
        "product production table",
        "product table",
        "product table header",
        "product table columns",
    },
    {
        "filling instructions",
        "instruções de preenchimento",
    },
    {
        "explanatory notes",
        "notas explicativas",
    },
]


def canonical_section(value):
    value_norm = normalise_identity_text(value)

    for group_index, group in enumerate(
        SECTION_EQUIVALENCE_GROUPS
    ):
        normalised_group = {
            normalise_identity_text(item)
            for item in group
        }

        if value_norm in normalised_group:
            return f"section_group_{group_index}"

    return value_norm


def normalise_code(value):
    if value is None:
        return None

    text = normalise_text(value)

    if not text:
        return None

    return text.upper()


def extract_pdf_page(value):
    text = normalise_text(value)

    match = re.search(
        r"(?:physical\s+)?pdf\s+page\s+(\d+)",
        text,
    )

    if not match:
        return None

    return int(match.group(1))


def description_similarity(a, b):
    a_norm = normalise_text(a)
    b_norm = normalise_text(b)

    if not a_norm and not b_norm:
        return 1.0

    if not a_norm or not b_norm:
        return 0.0

    return SequenceMatcher(
        None,
        a_norm,
        b_norm,
    ).ratio()


In [ ]:
# ============================================================
# 7. Deterministic one-to-one identity alignment
# ============================================================

def base_identity(record):
    return (
        normalise_identity_text(
            record["Category"]
        ),
        normalise_identity_text(
            record["Field or Concept"]
        ),
    )


def section_identity(record):
    return canonical_section(
        record["Section"]
    )


reference_records = (
    reference_df
    .to_dict("records")
)

extraction_records = (
    extracted_df
    .to_dict("records")
)


reference_groups = defaultdict(list)
extraction_groups = defaultdict(list)

for index, record in enumerate(reference_records):
    reference_groups[
        base_identity(record)
    ].append(index)

for index, record in enumerate(extraction_records):
    extraction_groups[
        base_identity(record)
    ].append(index)


matches = []
matched_reference = set()
matched_extraction = set()
ambiguous_alignment_groups = []


all_base_keys = sorted(
    set(reference_groups)
    | set(extraction_groups)
)


for key in all_base_keys:

    ref_indices = reference_groups.get(
        key,
        [],
    )

    ext_indices = extraction_groups.get(
        key,
        [],
    )

    if (
        len(ref_indices) == 1
        and len(ext_indices) == 1
    ):
        r_idx = ref_indices[0]
        e_idx = ext_indices[0]

        matches.append({
            "Reference Index": r_idx,
            "Extraction Index": e_idx,
            "Alignment Rule":
                "Category + Field or Concept",
        })

        matched_reference.add(r_idx)
        matched_extraction.add(e_idx)
        continue

    ref_by_section = defaultdict(list)
    ext_by_section = defaultdict(list)

    for r_idx in ref_indices:
        ref_by_section[
            section_identity(
                reference_records[r_idx]
            )
        ].append(r_idx)

    for e_idx in ext_indices:
        ext_by_section[
            section_identity(
                extraction_records[e_idx]
            )
        ].append(e_idx)


    for section_key in sorted(
        set(ref_by_section)
        | set(ext_by_section)
    ):

        r_list = ref_by_section.get(
            section_key,
            [],
        )

        e_list = ext_by_section.get(
            section_key,
            [],
        )

        if (
            len(r_list) == 1
            and len(e_list) == 1
        ):
            r_idx = r_list[0]
            e_idx = e_list[0]

            matches.append({
                "Reference Index": r_idx,
                "Extraction Index": e_idx,
                "Alignment Rule":
                    "Category + Field or Concept + canonical Section",
            })

            matched_reference.add(r_idx)
            matched_extraction.add(e_idx)

        elif r_list or e_list:
            ambiguous_alignment_groups.append({
                "base_identity": key,
                "canonical_section": section_key,
                "reference_indices": r_list,
                "extraction_indices": e_list,
            })


missing_reference_indices = sorted(
    set(range(len(reference_records)))
    - matched_reference
)

unsupported_extraction_indices = sorted(
    set(range(len(extraction_records)))
    - matched_extraction
)


print("Aligned:", len(matches))
print("Missing:", len(missing_reference_indices))
print(
    "Unsupported/unmatched:",
    len(unsupported_extraction_indices),
)
print(
    "Ambiguous identity groups:",
    len(ambiguous_alignment_groups),
)


In [ ]:
# ============================================================
# 8. Field comparison
# ============================================================

def exact_text_equal(a, b):
    return (
        normalise_text(a)
        == normalise_text(b)
    )


def section_equal(a, b):
    return (
        canonical_section(a)
        == canonical_section(b)
    )


def code_equal(a, b):
    return (
        normalise_code(a)
        == normalise_code(b)
    )


def source_location_equal(a, b):
    page_a = extract_pdf_page(a)
    page_b = extract_pdf_page(b)

    return (
        page_a is not None
        and page_b is not None
        and page_a == page_b
    )


comparison_rows = []


for match in matches:

    r = reference_records[
        match["Reference Index"]
    ]

    e = extraction_records[
        match["Extraction Index"]
    ]

    out = {
        "Reference Index":
            match["Reference Index"],

        "Extraction Index":
            match["Extraction Index"],

        "Alignment Rule":
            match["Alignment Rule"],
    }


    correctness = {
        "Category":
            exact_text_equal(
                r["Category"],
                e["Category"],
            ),

        "Section":
            section_equal(
                r["Section"],
                e["Section"],
            ),

        "Field or Concept":
            exact_text_equal(
                r["Field or Concept"],
                e["Field or Concept"],
            ),

        "Description":
            exact_text_equal(
                r["Description"],
                e["Description"],
            ),

        "Code":
            code_equal(
                r["Code"],
                e["Code"],
            ),

        "Expected Value Type":
            exact_text_equal(
                r["Expected Value Type"],
                e["Expected Value Type"],
            ),

        "Source Location":
            source_location_equal(
                r["Source Location"],
                e["Source Location"],
            ),
    }

    all_primary_fields_match = all(
        correctness[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    identity_fields_match = all(
        correctness[field]
        for field in ALIGNMENT_IDENTITY_FIELDS
    )

    identity_label_difference = (
        all_primary_fields_match
        and not identity_fields_match
    )


    for field in FIELDS:
        out[
            f"Reference {field}"
        ] = r[field]

        out[
            f"Extracted {field}"
        ] = e[field]

        out[
            f"{field} Correct"
        ] = bool(
            correctness[field]
        )


    out[
        "Description Similarity Diagnostic"
    ] = float(
        description_similarity(
            r["Description"],
            e["Description"],
        )
    )


    out[
        "Fully Correct Primary Record"
    ] = bool(
        all_primary_fields_match
    )

    out[
        "Identity Fields Match"
    ] = bool(
        identity_fields_match
    )

    out[
        "Identity Label Difference"
    ] = bool(
        identity_label_difference
    )


    comparison_rows.append(out)


comparison_df = pd.DataFrame(
    comparison_rows
)


missing_records_df = (
    reference_df
    .iloc[
        missing_reference_indices
    ]
    .copy()
)


unsupported_records_df = (
    extracted_df
    .iloc[
        unsupported_extraction_indices
    ]
    .copy()
)


discrepant_records_df = (
    comparison_df.loc[
        ~comparison_df[
            "Fully Correct Primary Record"
        ]
    ]
    .copy()
)


print(
    "Fully correct primary records:",
    int(
        comparison_df[
            "Fully Correct Primary Record"
        ].sum()
    )
)

print(
    "Primary discrepant records:",
    len(discrepant_records_df),
)


In [ ]:
# ============================================================
# 9. Metrics
# ============================================================

aligned_records = len(
    comparison_df
)

fully_correct_records = int(
    comparison_df[
        "Fully Correct Primary Record"
    ].sum()
)

discrepant_records = (
    aligned_records
    - fully_correct_records
)

missing_records = len(
    missing_reference_indices
)

unsupported_records = len(
    unsupported_extraction_indices
)


completeness = (
    aligned_records
    / len(reference_df)
    if len(reference_df)
    else 0.0
)

record_precision_exact = (
    fully_correct_records
    / len(extracted_df)
    if len(extracted_df)
    else 0.0
)

record_recall_exact = (
    fully_correct_records
    / len(reference_df)
    if len(reference_df)
    else 0.0
)

record_f1_exact = (
    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )
    if (
        record_precision_exact
        + record_recall_exact
    )
    else 0.0
)


primary_field_accuracy = {}

for field in PRIMARY_CORRECTNESS_FIELDS:

    primary_field_accuracy[field] = float(
        comparison_df[
            f"{field} Correct"
        ].mean()
    ) if aligned_records else 0.0


diagnostic_field_accuracy = {
    "Section": float(
        comparison_df[
            "Section Correct"
        ].mean()
    ) if aligned_records else 0.0,

    "Description exact": float(
        comparison_df[
            "Description Correct"
        ].mean()
    ) if aligned_records else 0.0,

    "Description mean lexical similarity": float(
        comparison_df[
            "Description Similarity Diagnostic"
        ].mean()
    ) if aligned_records else 0.0,
}


field_accuracy = (
    sum(
        primary_field_accuracy.values()
    )
    / len(primary_field_accuracy)
    if primary_field_accuracy
    else 0.0
)

category_metrics = {}

for category, expected in (
    EXPECTED_CATEGORY_COUNTS.items()
):

    ref_subset = reference_df[
        reference_df["Category"]
        == category
    ]

    ext_subset = extracted_df[
        extracted_df["Category"]
        == category
    ]

    aligned_subset = comparison_df[
        comparison_df[
            "Reference Category"
        ]
        == category
    ]

    full = int(
        aligned_subset[
            "Fully Correct Primary Record"
        ].sum()
    )

    category_metrics[category] = {
        "expected_records": int(expected),
        "extracted_records": int(
            len(ext_subset)
        ),
        "aligned_records": int(
            len(aligned_subset)
        ),
        "fully_correct_records": full,
        "discrepant_records": int(
            len(aligned_subset) - full
        ),
        "completeness": (
            len(aligned_subset)
            / expected
            if expected
            else 0.0
        ),
        "record_precision_exact": (
            full
            / len(ext_subset)
            if len(ext_subset)
            else 0.0
        ),
        "record_recall_exact": (
            full
            / len(ref_subset)
            if len(ref_subset)
            else 0.0
        ),
    }

    p = category_metrics[
        category
    ][
        "record_precision_exact"
    ]

    r = category_metrics[
        category
    ][
        "record_recall_exact"
    ]

    category_metrics[
        category
    ][
        "record_f1_exact"
    ] = (
        2 * p * r / (p + r)
        if p + r
        else 0.0
    )


print("Reference records:", len(reference_df))
print("Extracted records:", len(extracted_df))
print("Aligned records:", aligned_records)
print("Fully correct records:", fully_correct_records)
print("Discrepant records:", discrepant_records)
print("Missing records:", missing_records)
print("Unsupported/unmatched records:", unsupported_records)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Field accuracy:",
    round(field_accuracy, 4),
)
print("Schema valid:", schema_validity)


In [ ]:
# ============================================================
# 10. Build final validation summary
# ============================================================

VALIDATION_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "input_representation": INPUT_REPRESENTATION,

    "reference_records":
        int(len(reference_df)),

    "extracted_records":
        int(len(extracted_df)),

    "aligned_records":
        int(aligned_records),

    "fully_correct_records":
        int(fully_correct_records),

    "discrepant_records":
        int(discrepant_records),

    "missing_records":
        int(missing_records),

    "unsupported_extracted_records":
        int(unsupported_records),

    "completeness":
        float(completeness),

    "missing_rate":
        float(
            missing_records
            / len(reference_df)
            if len(reference_df)
            else 0.0
        ),

    "record_precision_exact":
        float(record_precision_exact),

    "record_recall_exact":
        float(record_recall_exact),

    "record_f1_exact":
        float(record_f1_exact),

    "unsupported_rate":
        float(
            unsupported_records
            / len(extracted_df)
            if len(extracted_df)
            else 0.0
        ),

    "discrepancy_rate_among_aligned":
        float(
            discrepant_records
            / aligned_records
            if aligned_records
            else 0.0
        ),

    "field_accuracy":
        float(
            field_accuracy
        ),

    "primary_field_accuracy":
        primary_field_accuracy,

    "diagnostic_field_accuracy":
        diagnostic_field_accuracy,

    "schema_validity":
        bool(schema_validity),

    "schema_diagnostics": {
        "top_level_object_valid":
            isinstance(
                parsed_extraction,
                dict,
            ),

        "document_id_correct":
            bool(document_id_correct),

        "branch_correct":
            bool(branch_correct),

        "records_is_list":
            bool(records_is_list),

        "record_schema_valid":
            bool(
                extraction_schema_exact
            ),

        "field_types_valid":
            bool(
                extraction_types_valid
            ),

        "schema_validity":
            bool(schema_validity),
    },

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "content_diagnostics": {
        "reference_record_count_valid":
            bool(
                reference_record_count_valid
            ),

        "reference_category_counts_valid":
            bool(
                reference_category_counts_valid
            ),

        "extraction_record_count_valid":
            bool(
                extraction_record_count_valid
            ),

        "extraction_category_counts_valid":
            bool(
                extraction_category_counts_valid
            ),

        "ambiguous_alignment_group_count":
            int(
                len(
                    ambiguous_alignment_groups
                )
            ),
    },

    "matching_rules": {
        "base_identity_fields":
            BASE_IDENTITY_FIELDS,

        "duplicate_disambiguation_field":
            DUPLICATE_DISAMBIGUATION_FIELD,

        "alignment_identity_fields":
            ALIGNMENT_IDENTITY_FIELDS,

        "one_to_one_assignment":
            (
                "Deterministic Category + Field or Concept identity; "
                "canonical Section used only when the same concept label "
                "occurs more than once within a category."
            ),

        "code_used_for_alignment":
            False,

        "expected_value_type_used_for_alignment":
            False,

        "description_used_for_alignment":
            False,

        "source_location_used_for_alignment":
            False,
        "equivalence_rules_frozen":
            True
    },

    "comparison_rules": {
        "raw_extraction_modified":
            False,

        "manual_correction_applied":
            False,

        "comparison_normalisation_scope":
            "Comparison copies only",

        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,

        "section":
            (
                "Canonical source-grounded section diagnostic; "
                "excluded from primary exact-record correctness because "
                "the prompt permits concise stable section labels."
            ),

        "description":
            (
                "Normalised exact diagnostic plus lexical-similarity "
                "diagnostic; excluded from primary exact-record "
                "correctness because the prompt permits a concise "
                "source-grounded description."
            ),

        "code":
            (
                "Exact printed-code agreement after conservative "
                "case/whitespace normalisation; null must remain null."
            ),

        "expected_value_type":
            (
                "Normalised exact textual agreement; no fuzzy "
                "correctness and no identifier/numeric-identifier "
                "equivalence."
            ),

        "source_location":
            (
                "Correct physical PDF page required. Source-region "
                "wording is not required to be identical because the "
                "prompt requests a concise region description."
            ),

        "d10_equivalence_rules_status": (
            "Frozen D10 document/schema-level comparison rules. "
            "No additional Branch-A-derived semantic equivalence rules "
            "were required. The identifier/numeric-identifier distinction "
            "is retained as semantically meaningful. "
            "Reuse unchanged for Branches A, B and C."
        )
    },

    "reference_integrity_confirmation": {
        "reference_semantics_valid":
            bool(
                reference_semantics_valid
            ),

        "checks":
            reference_semantic_checks,

        "reference_modified_by_validation":
            False,
    },

    "category_metrics":
        category_metrics,

    "input_provenance": {
        "reference_file":
            REFERENCE_PATH.name,

        "reference_sha256":
            REFERENCE_SHA256,

        "parsed_extraction_file":
            EXTRACTION_PATH.name,

        "parsed_extraction_sha256":
            EXTRACTION_SHA256,

        "technical_diagnostics_file":
            TECHNICAL_DIAGNOSTICS_PATH.name,

        "technical_diagnostics_sha256":
            TECHNICAL_DIAGNOSTICS_SHA256,

        "experiment_metadata_file":
            METADATA_PATH.name,

        "experiment_metadata_sha256":
            METADATA_SHA256,

        "branch_a_structurally_evaluable":
            bool(
                branch_a_structurally_evaluable
            ),

        "parsed_extraction_hash_matches_metadata":
            bool(
                parsed_extraction_hash_matches_metadata
            ),

        "source_hash_matches_stage_1":
            bool(
                source_hash_matches_stage_1
            ),
    },

}


print(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2,
    )
)

assert (
    fully_correct_records
    + discrepant_records
    == aligned_records
)


In [ ]:
# ============================================================
# 11. Export reproducible validation outputs
# ============================================================

SUMMARY_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_validation_summary.json"
)

DETAILED_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_validation_detailed.csv"
)

DISCREPANT_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_discrepant_records.csv"
)

MISSING_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_missing_records.csv"
)

UNSUPPORTED_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_unsupported_records.csv"
)

REFERENCE_SEMANTICS_PATH = (
    OUTPUT_DIR
    / "D10_reference_semantics_confirmation.json"
)

ALIGNMENT_ISSUES_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_alignment_issues.json"
)


SUMMARY_PATH.write_text(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig",
)

discrepant_records_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_records_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig",
)

unsupported_records_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig",
)

REFERENCE_SEMANTICS_PATH.write_text(
    json.dumps(
        {
            "document_id":
                DOCUMENT_ID,

            "reference_semantics_valid":
                reference_semantics_valid,

            "checks":
                reference_semantic_checks,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

ALIGNMENT_ISSUES_PATH.write_text(
    json.dumps(
        ambiguous_alignment_groups,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

assert reference_semantics_valid
assert branch_a_structurally_evaluable
assert parsed_extraction_hash_matches_metadata
assert source_hash_matches_stage_1
assert schema_validity

required_outputs = [
    SUMMARY_PATH,
    DETAILED_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    REFERENCE_SEMANTICS_PATH,
    ALIGNMENT_ISSUES_PATH,
]

missing_outputs = [
    path.name
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:
    raise AssertionError(
        f"Missing output files: {missing_outputs}"
    )


print("Validation A — D10 completed.")
print(
    "Reference semantics valid:",
    reference_semantics_valid,
)
print(
    "Branch A structurally evaluable:",
    branch_a_structurally_evaluable,
)
print(
    "Aligned / reference:",
    f"{aligned_records}/{len(reference_df)}",
)
print(
    "Fully correct primary records:",
    fully_correct_records,
)
print(
    "Discrepant primary records:",
    discrepant_records,
)
print(
    "Missing records:",
    missing_records,
)
print(
    "Unsupported/unmatched records:",
    unsupported_records,
)
print(
    "Exact F1:",
    round(record_f1_exact, 4),
)
print(
    "Field accuracy:",
    round(
        field_accuracy,
        4,
    ),
)
print(
    "Equivalence rules frozen:",
    True,
)
print(
    "D10 comparison rules are frozen. "
    "Reuse the same Stage 1 reference, identity alignment, "
    "normalisation and correctness rules unchanged for "
    "Branches B and C."
)

for path in required_outputs:
    print("-", path.name)
